In [2]:
# ============================================================
# 🖼 Stable Diffusion Image-to-Image (Realistic + Clear Output)
# ============================================================

# STEP 1 — Install dependencies
!pip install -q diffusers transformers accelerate safetensors huggingface-hub xformers gradio

# STEP 2 — Check GPU
import torch, os
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("⚠ Go to Runtime → Change runtime type → GPU → Save")

# STEP 3 — Mount Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/colab_img2img_outputs"
os.makedirs(output_dir, exist_ok=True)
print("✅ Drive mounted. Outputs →", output_dir)

# STEP 4 — Hugging Face authentication
# Use Colab Secrets to hide your token
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
  raise ValueError("HF_TOKEN not found in Colab Secrets. Please add it to the Secrets tab.")

# STEP 5 — Load Realistic Vision model for sharper results
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "SG161222/Realistic_Vision_V5.1_noVAE"  # sharper, more realistic model

print("⏳ Loading model ...")
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    token=HF_TOKEN
).to(device)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("✅ xFormers enabled (fast + memory efficient).")
except Exception as e:
    print("⚠ xFormers not enabled:", e)

print("✅ Model ready on", device)

# STEP 6 — Upload your input image
from google.colab import files
print("📤 Upload an image (e.g. room.jpg, livingroom.png)")
uploaded = files.upload()
input_image_path = list(uploaded.keys())[0]
init_image = Image.open(input_image_path).convert("RGB").resize((512, 512))
init_image

# STEP 7 — Generate a clearer, detailed image
prompt = (
    "a realistic modern bedroom interior with stylish wooden furniture, "
    "cozy lighting, ultra sharp details, 8k architectural photography, "
    "depth of field, photorealistic"
)
strength = 0.45          # 0.4–0.5 keeps structure & improves clarity
guidance = 8.0           # prompt adherence
steps = 45               # higher = more detail

print("✨ Generating high-clarity image ...")
result = pipe(
    prompt=prompt,
    image=init_image,
    strength=strength,
    guidance_scale=guidance,
    num_inference_steps=steps
).images[0]

output_path = os.path.join(output_dir, "img2img_highclarity.png")
result.save(output_path)
print("✅ Saved:", output_path)
result.show()

# STEP 8 — Optional: Gradio interface for teammates
import gradio as gr

def generate_img2img(input_img, prompt, strength=0.45, guidance=8.0, steps=45):
    res = pipe(
        prompt=prompt,
        image=input_img,
        strength=strength,
        guidance_scale=guidance,
        num_inference_steps=steps
    ).images[0]
    return res

demo = gr.Interface(
    fn=generate_img2img,
    inputs=[
        gr.Image(type="pil", label="Input Image"),
        gr.Textbox(label="Prompt", value="a modern bedroom interior, high detail, 8k realism"),
        gr.Slider(0.1, 1.0, value=0.45, step=0.05, label="Transformation Strength"),
    ],
    outputs=gr.Image(label="Generated Image"),
    title="🎨 Image-to-Image Generator — High Clarity",
    description="Upload an image + describe how to transform it (uses Realistic Vision 5.1 model)."
)
demo.launch(share=True)

Torch version: 2.8.0+cu126
CUDA available: True
GPU device: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Outputs → /content/drive/MyDrive/colab_img2img_outputs


SecretNotFoundError: Secret HF_TOKEN does not exist.